# PSMamba on Kaggle: Setup and Smoke Test (v2, no compiling)

**Why v2:** Kaggle now runs **Python 3.13, torch 2.11, CUDA 13**. The official `mamba-ssm` has no prebuilt wheel for that and fails to compile. So this version uses **`mambapy`**, a pure-PyTorch Mamba. It installs in seconds and works on any torch. It is slower and uses more memory than the official CUDA kernels, but it is fine for 196-token images and for getting the project moving.

**Before you run:** Accelerator = **GPU T4** and Internet = **On** (start the session first, then pick the GPU).

## 1. Check the environment

In [ ]:
import sys, torch
print("Python:", sys.version.split()[0])
print("Torch :", torch.__version__, "| CUDA:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {p.name} | {p.total_memory/1e9:.1f} GB | compute capability {p.major}.{p.minor}")
else:
    print("NO GPU. Turn on the accelerator, then re-run. (The tests below still run on CPU, just slowly.)")

## 2. Install (takes a few seconds)

In [ ]:
!pip install -q mambapy
import mambapy, importlib.metadata as md
print("mambapy", md.version("mambapy"))

## 3. Test A: one Mamba block, forward + backward

Runs in fp32 first (safest), then tries fp16 autocast. If the fp16 part fails, just use fp32 for now and tell me.

In [ ]:
import torch
from mambapy.mamba import MambaBlock, MambaConfig

dev = "cuda" if torch.cuda.is_available() else "cpu"
cfg = MambaConfig(d_model=192, n_layers=1, d_state=16, expand_factor=2, d_conv=4, pscan=True, use_cuda=False)
blk = MambaBlock(cfg).to(dev)

# fp32
x = torch.randn(8, 196, 192, device=dev, requires_grad=True)
y = blk(x)
loss = y.pow(2).mean(); loss.backward()
print("fp32 OK | out:", tuple(y.shape), "| loss:", round(loss.item(), 4),
      "| grad finite:", torch.isfinite(x.grad).all().item())

# fp16 autocast (optional)
if dev == "cuda":
    try:
        x2 = torch.randn(8, 196, 192, device=dev, requires_grad=True)
        with torch.autocast("cuda", dtype=torch.float16):
            y2 = blk(x2)
        l2 = y2.float().pow(2).mean(); l2.backward()
        print("fp16 autocast OK | loss:", round(l2.item(), 4), "| grad finite:", torch.isfinite(x2.grad).all().item())
    except Exception as e:
        print("fp16 autocast FAILED, use fp32 for now:", repr(e)[:300])

## 4. Test B: small bidirectional Mamba encoder on 224 / 160 / 96 crops

Simplified from the paper's Eq. 7: a forward scan and a backward scan, fused by learnable weights. This is a stand-in for the paper's encoder, not the official Vision Mamba code. With patch size 16 the token grids are 14x14, 10x10 and 6x6.

In [ ]:
import time, torch, torch.nn as nn, torch.nn.functional as F
from mambapy.mamba import MambaBlock, MambaConfig

def make_block(d):
    return MambaBlock(MambaConfig(d_model=d, n_layers=1, d_state=16, expand_factor=2,
                                  d_conv=4, pscan=True, use_cuda=False))

class BiMambaBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.fwd = make_block(d)
        self.bwd = make_block(d)
        self.wf = nn.Parameter(torch.tensor(0.5))   # learnable fusion weights (paper Eq. 7)
        self.wb = nn.Parameter(torch.tensor(0.5))
    def forward(self, x):
        h = self.norm(x)
        f = self.fwd(h)
        b = self.bwd(h.flip(1)).flip(1)
        return x + self.wf * f + self.wb * b

class MiniBiMamba(nn.Module):
    def __init__(self, d=192, depth=6, patch=16, base_grid=14):
        super().__init__()
        self.embed = nn.Conv2d(3, d, patch, patch)
        self.pos = nn.Parameter(torch.zeros(1, d, base_grid, base_grid))
        nn.init.trunc_normal_(self.pos, std=0.02)
        self.blocks = nn.ModuleList([BiMambaBlock(d) for _ in range(depth)])
        self.norm = nn.LayerNorm(d)
    def forward(self, img):
        t = self.embed(img)                                   # B, d, gh, gw
        gh, gw = t.shape[-2:]
        pos = F.interpolate(self.pos, size=(gh, gw), mode="bicubic", align_corners=False)
        t = (t + pos).flatten(2).transpose(1, 2)              # B, N, d
        for blk in self.blocks:
            t = blk(t)
        return self.norm(t), (gh, gw)

dev = "cuda" if torch.cuda.is_available() else "cpu"
model = MiniBiMamba().to(dev)
print("Params (M):", round(sum(p.numel() for p in model.parameters()) / 1e6, 2))

BATCH = 8   # lower this to 4 if you get an out-of-memory error
for size in (224, 160, 96):
    if dev == "cuda":
        torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    x = torch.randn(BATCH, 3, size, size, device=dev)
    t0 = time.time()
    tokens, grid = model(x)
    tokens.pow(2).mean().backward()
    if dev == "cuda":
        torch.cuda.synchronize()
        mem = f"{torch.cuda.max_memory_allocated()/1e9:.2f} GB"
    else:
        mem = "n/a (cpu)"
    print(f"{size}x{size}: grid {grid}, tokens {tuple(tokens.shape)}, {time.time()-t0:.2f}s, peak mem {mem}")
print("Test B OK")

## 5. Helpers: seed and checkpoint/resume

Kaggle sessions can die, so save often and resume from `/kaggle/working`.

In [ ]:
import os, random, numpy as np, torch

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

CKPT = "/kaggle/working/ckpt.pt"

def save_ckpt(state, path=CKPT):
    tmp = path + ".tmp"
    torch.save(state, tmp)
    os.replace(tmp, path)          # atomic, so a crash never leaves a half-written file

def load_ckpt(path=CKPT):
    return torch.load(path, map_location="cpu") if os.path.exists(path) else None

set_seed(42)
print("helpers ready | existing checkpoint:", os.path.exists(CKPT))

## 6. Find your datasets

Add PlantDoc and Citrus via **Add Input** (search "PlantDoc", "citrus leaves"). They appear under `/kaggle/input`.

In [ ]:
import os
root = "/kaggle/input"
if not os.path.isdir(root) or not os.listdir(root):
    print("No datasets added yet. Use Add Input in the sidebar.")
else:
    for name in sorted(os.listdir(root)):
        p = os.path.join(root, name)
        n = sum(len(f) for _, _, f in os.walk(p))
        print(f"{name}: {n} files")
        for sub in sorted(os.listdir(p))[:8]:
            print("   -", sub)

## What to check

- Test A: `fp32 OK` (and ideally `fp16 autocast OK`)
- Test B: three lines with grid, time and memory, then `Test B OK`
- Your datasets are listed

Later, once everything else works, we can revisit the official CUDA kernels for speed (optional).